In [ ]:

from google.colab import drive
drive.mount('/content/drive')

%pip install brian2 brian2hears librosa numpy scipy matplotlib setuptools

import logging
logging.getLogger('brian2').setLevel(logging.ERROR)

import brian2
brian2.prefs.codegen.target = 'numpy'

import numpy as np
import os
import glob
import random
import tensorflow as tf
from scipy import signal
from math import gcd
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, CSVLogger
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


output_dir = "/content/drive/MyDrive/p&d/fase_2/"
os.makedirs(output_dir, exist_ok=True)
print(f" Map aangemaakt/bevestigd: {output_dir}")

PAD_TRAIN_EEG   = "/content/drive/MyDrive/p&d/fase_2/data/64 Channel Biosemi unprocessed data - train/*/*/*.npz"
PAD_TEST_EEG    = "/content/drive/MyDrive/p&d/fase_2/data/test_data/test_data/*/*.npz"
PAD_AUDIO       = "/content/drive/MyDrive/p&d/fase_2/data/preprocessed/audio/cnn/"

PAD_CHECKPOINT  = "/content/drive/MyDrive/p&d/fase_2/hybrid_dilated_lstm_CHECKPOINT_{epoch:02d}_{val_accuracy:.4f}.keras"
PAD_BEST_MODEL  = "/content/drive/MyDrive/p&d/fase_2/hybrid_dilated_lstm_BEST.keras"
PAD_EIND_MODEL  = "/content/drive/MyDrive/p&d/fase_2/hybrid_dilated_lstm_EIND.keras"
PAD_CSV_LOG     = "/content/drive/MyDrive/p&d/fase_2/training_log_hybrid_dilated_lstm.csv"
PAD_HISTORY_NPY = "/content/drive/MyDrive/p&d/fase_2/training_history_hybrid_dilated_lstm.npy"

def process_eeg_file(npz_filename, mode='cnn'):
    data           = np.load(npz_filename)
    eeg_data       = data['eeg']
    fs             = int(data['fs'])
    attended_wav   = str(data['stimulus_attended'])
    unattended_wav = str(data['stimulus_unattended'])

    if mode == 'cnn':
        target_sr, lowcut, highcut = 64, 1.0, 32.0
    elif mode == 'linear':
        target_sr, lowcut, highcut = 20, 1.0, 9.0
    else:
        raise ValueError("Kies 'linear' of 'cnn'")

    sos            = signal.butter(N=4, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')
    eeg_filtered   = signal.sosfiltfilt(sos, eeg_data, axis=0)
    g              = gcd(fs, target_sr)
    eeg_downsampled = signal.resample_poly(eeg_filtered, target_sr // g, fs // g, axis=0)

    return eeg_downsampled, attended_wav, unattended_wav

def batch_equalizer(eeg, env_1, env_2, labels):
    return (
        np.concatenate([eeg,   eeg  ], axis=0),
        np.concatenate([env_1, env_2], axis=0),
        np.concatenate([env_2, env_1], axis=0)
    ), np.concatenate([labels, (labels + 1) % 2], axis=0)


class DataGenerator:
    def __init__(self, files, audio_dir, time_window=640):
        self.files       = files
        self.audio_dir   = audio_dir
        self.time_window = time_window

        alle_audio       = glob.glob(os.path.join(self.audio_dir, "**", "*.npy"), recursive=True)
        self.audio_dict  = {os.path.basename(f).lower(): f for f in alle_audio}
        print(f" {len(self.audio_dict)} audiobestanden gevonden")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        eeg_processed, att_naam, unatt_naam = process_eeg_file(self.files[idx], mode='cnn')

        zoek_att   = f"{att_naam.replace('.wav','').replace('.npy','').strip().lower()}_cnn.npy"
        zoek_unatt = f"{unatt_naam.replace('.wav','').replace('.npy','').strip().lower()}_cnn.npy"

        att_pad   = self.audio_dict.get(zoek_att)
        unatt_pad = self.audio_dict.get(zoek_unatt)
        if not att_pad or not unatt_pad:
            raise FileNotFoundError(f"Audio '{zoek_att}' of '{zoek_unatt}' niet gevonden!")

        env1 = np.load(att_pad)
        env2 = np.load(unatt_pad)

        min_len = min(eeg_processed.shape[0], env1.shape[0], env2.shape[0])
        eeg_processed = eeg_processed[:min_len]
        env1 = env1[:min_len]
        env2 = env2[:min_len]

        n = min_len // self.time_window
        eeg_w  = np.array(np.split(eeg_processed[:n * self.time_window], n))
        env1_w = np.expand_dims(np.array(np.split(env1[:n * self.time_window], n)), axis=-1)
        env2_w = np.expand_dims(np.array(np.split(env2[:n * self.time_window], n)), axis=-1)

        labels = np.ones((n, 1))
        return batch_equalizer(eeg_w, env1_w, env2_w, labels)

    def __call__(self):
        for idx in range(len(self)):
            try:
                yield self.__getitem__(idx)
            except Exception as e:
                print(f"  Fout bij trial {idx}: {e}")
                continue
            if idx == len(self) - 1:
                np.random.shuffle(self.files)


def maak_tf_dataset(files, audio_dir, time_window=640):
    gen = DataGenerator(files=files, audio_dir=audio_dir, time_window=time_window)
    return tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            (tf.TensorSpec(shape=(None, time_window, 64), dtype=tf.float32),
             tf.TensorSpec(shape=(None, time_window,  1), dtype=tf.float32),
             tf.TensorSpec(shape=(None, time_window,  1), dtype=tf.float32)),
            tf.TensorSpec(shape=(None, 1), dtype=tf.float32)
        )
    )


def bouw_hybrid_model(
    time_window    = 640,   
    n_layers       = 3,     
    kernel_size    = 3,     
    spatial_filters = 8,   
    dilated_filters = 16, 
    lstm_units     = 16,   
):
    eeg  = tf.keras.layers.Input(shape=[time_window, 64], name="EEG_Input")
    env1 = tf.keras.layers.Input(shape=[time_window,  1], name="Env1_Input")
    env2 = tf.keras.layers.Input(shape=[time_window,  1], name="Env2_Input")

    x = tf.keras.layers.BatchNormalization(name="EEG_BN_Input")(eeg)

    x = tf.keras.layers.Conv1D(spatial_filters, kernel_size=1,
                                padding='same', name="Spatial_Conv")(x)
    x = tf.keras.layers.BatchNormalization(name="EEG_BN_Spatial")(x)

    for l in range(n_layers):
        dil = kernel_size ** l
        x = tf.keras.layers.Conv1D(
            dilated_filters, kernel_size=kernel_size,
            dilation_rate=dil, padding='same',
            activation='relu', name=f"EEG_Dilated_L{l}_D{dil}"
        )(x)
        x = tf.keras.layers.BatchNormalization(name=f"EEG_BN_Dilated_L{l}")(x)

    eeg_out = x   

    a1 = tf.keras.layers.BatchNormalization(name="Audio_BN_Input")(env1)
    a2 = tf.keras.layers.BatchNormalization(name="Audio_BN_Input_2")(env2)

    for l in range(n_layers):
        dil = kernel_size ** l
        shared_conv = tf.keras.layers.Conv1D(
            dilated_filters, kernel_size=kernel_size,
            dilation_rate=dil, padding='same',
            activation='relu', name=f"Audio_Dilated_L{l}_D{dil}"
        )
        shared_bn = tf.keras.layers.BatchNormalization(name=f"Audio_BN_Dilated_L{l}")

        a1 = shared_bn(shared_conv(a1))
        a2 = shared_bn(shared_conv(a2))

    shared_lstm = tf.keras.layers.LSTM(
        lstm_units, return_sequences=True,
        activation='tanh', name="Audio_LSTM_Shared"
    )
    audio_out1 = shared_lstm(a1)   # (batch, time_window, lstm_units)
    audio_out2 = shared_lstm(a2)

    cos1 = tf.keras.layers.Dot(axes=1, normalize=True, name="Cosine_Env1")([eeg_out, audio_out1])
    cos2 = tf.keras.layers.Dot(axes=1, normalize=True, name="Cosine_Env2")([eeg_out, audio_out2])

    
    flat  = tf.keras.layers.Flatten()(tf.keras.layers.Concatenate()([cos1, cos2]))
    out1  = tf.keras.layers.Dense(1, activation='sigmoid', name='Decision_Neuron')(flat)
    out   = tf.keras.layers.Reshape([1], name='output_name')(out1)

    model = tf.keras.Model(inputs=[eeg, env1, env2], outputs=[out])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


TIME_WINDOW = 640   

alle_eeg = glob.glob(PAD_TRAIN_EEG)
print(f" Totaal EEG trials gevonden: {len(alle_eeg)}")

random.seed(42)
random.shuffle(alle_eeg)
split_idx   = int(len(alle_eeg) * 0.8)
train_files = alle_eeg[:split_idx]
val_files   = alle_eeg[split_idx:]
print(f"   Train: {len(train_files)} | Val: {len(val_files)}")

train_dataset = maak_tf_dataset(train_files, PAD_AUDIO, time_window=TIME_WINDOW)
val_dataset   = maak_tf_dataset(val_files,   PAD_AUDIO, time_window=TIME_WINDOW)

model = bouw_hybrid_model(time_window=TIME_WINDOW, n_layers=3)
model.summary()


checkpoint_epoch = ModelCheckpoint(
    filepath=PAD_CHECKPOINT,   
    monitor='val_accuracy',
    save_best_only=True,       
    save_weights_only=False,   
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

csv_logger = CSVLogger(PAD_CSV_LOG, append=False)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=[checkpoint_epoch, early_stop, csv_logger]
)

model.save(PAD_EIND_MODEL)
np.save(PAD_HISTORY_NPY, history.history)
print(f"\n Eindmodel opgeslagen: {PAD_EIND_MODEL}")
print(f" Trainingsgeschiedenis opgeslagen: {PAD_HISTORY_NPY}")


test_files = glob.glob(PAD_TEST_EEG)
print(f"   Test-trials gevonden: {len(test_files)}")

if len(test_files) == 0:
    print("  Geen testbestanden gevonden")
else:
    test_dataset = maak_tf_dataset(test_files, PAD_AUDIO, time_window=TIME_WINDOW)

    best_model = tf.keras.models.load_model(PAD_BEST_MODEL) if os.path.exists(PAD_BEST_MODEL) else model

    test_loss, test_acc = best_model.evaluate(test_dataset)
 
    print(f"  Test Loss     : {test_loss:.4f}")
    print(f"  Test Accuracy : {test_acc * 100:.2f}%")

acc       = history.history['accuracy']
val_acc   = history.history['val_accuracy']
loss_hist = history.history['loss']
val_loss  = history.history['val_loss']
epochs_r  = range(1, len(acc) + 1)

best_epoch    = int(np.argmax(val_acc)) + 1
best_val_acc  = val_acc[best_epoch - 1]
best_val_loss = val_loss[best_epoch - 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hybrid Dilated-LSTM Model — Trainingsresultaten', fontsize=15, fontweight='bold')

ax1.plot(epochs_r, acc,     'b-', label='Train Accuracy',      linewidth=2)
ax1.plot(epochs_r, val_acc, 'g-', label='Validation Accuracy', linewidth=2)
ax1.axvline(x=best_epoch, color='red', linestyle='--', label=f'Best Epoch ({best_epoch})', alpha=0.8)
ax1.scatter(best_epoch, best_val_acc, color='red', s=60, zorder=5)
ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax1.text(0.05, 0.95,
         f" Best Val Accuracy: {best_val_acc*100:.2f}%\n(Epoch {best_epoch})",
         transform=ax1.transAxes, fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='red', alpha=0.9))
ax1.set_title('A. Accuracy', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='lower right')

ax2.plot(epochs_r, loss_hist, 'b-', label='Train Loss',      linewidth=2)
ax2.plot(epochs_r, val_loss,  'g-', label='Validation Loss', linewidth=2)
ax2.axvline(x=best_epoch, color='red', linestyle='--', label=f'Best Epoch ({best_epoch})', alpha=0.8)
ax2.scatter(best_epoch, best_val_loss, color='red', s=60, zorder=5)
ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax2.text(0.05, 0.05,
         f" Lowest Val Loss: {best_val_loss:.4f}\n(Epoch {best_epoch})",
         transform=ax2.transAxes, fontsize=11, verticalalignment='bottom',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='red', alpha=0.9))
ax2.set_title('B. Loss', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Binary Cross-Entropy Loss')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/fase_2/hybrid_dilated_lstm_training_plot.png', dpi=150)
plt.show()
